In [ ]:
#!/usr/bin/env python
# coding: utf-8

## Setup

In[1]:

In [ ]:
import regex as re
from collections import defaultdict
from tqdm.contrib.concurrent import process_map

## Step 1, chunk text by special tokens

In[2]:

In [ ]:
def split_by_special(text,
                     special_tokens, # list of special tokens, e.g., ["<|endoftext|>", "<|pad|>", "<|unk|>"]
                     drop_special=True, # whether or not drop the special tokens when using them to split
                     ):
    PAT = "|".join(re.escape(tok) for tok in special_tokens)
    if not drop_special: PAT = f'({PAT})' # capturing group to keep special tokens
    return re.split(PAT, text)

In[3]:

In [ ]:
special_tokens = ["<|endoftext|>", "<|pad|>", "<|unk|>","<|endoftext|><|endoftext|>"]

In[4]:

In [ ]:
test = "<|pad|>abc<|pad|>"

In[5]:

In [ ]:
split_by_special(test,special_tokens,drop_special=False)

need to solve the above as there's nothing in the edge

In[6]:

In [ ]:
def split_by_special(text, special_tokens, drop_special=True):
    if not special_tokens:  # if there's no special tokens, return the whole text
        return [text]
    PAT = "|".join(re.escape(tok) for tok in special_tokens)
    if not drop_special: PAT = f"({PAT})"  # capture group to keep special tokens
    chunks = re.split(PAT, text)
    return [c for c in chunks if c]  # remove empty strings

In[7]:

In [ ]:
split_by_special(test,special_tokens,drop_special=False)

In[8]:

In [ ]:
test = "Hello, how <|endoftext|><|endoftext|> are you?<|endoftext|>"

In[9]:

In [ ]:
split_by_special(test,special_tokens,drop_special=False)

<|endoftext|> appear twice but not as a whole

In[10]:

In [ ]:
def split_by_special(text, special_tokens, drop_special=True):
    if not special_tokens:
        return [text]

    # Sort by descending length to prioritize longer tokens (e.g., "<|endoftext|><|endoftext|>" before "<|endoftext|>")
    special_tokens = sorted(special_tokens, key=len, reverse=True)
    PAT = "|".join(re.escape(tok) for tok in special_tokens)
    if not drop_special: PAT = f"({PAT})"
    chunks = re.split(PAT, text)
    return [c for c in chunks if c]  # remove empty strings

In[11]:

In [ ]:
split_by_special(test,special_tokens,drop_special=False)
# now appear as a whole

In[12]:

In [ ]:
test =  "Hello, world!<|pad|> Hello, world. <|endoftext|>How are you? <|unk|> I'm fine, thank you! And you?"

In[13]:

In [ ]:
chunks = split_by_special(test,special_tokens)
chunks

In[14]:

In [ ]:
chunk=chunks[0]
chunk

## Step 2, pre-tokenize: split chunk into word list by GPT2 pattern and get word counts

In[15]:

In [ ]:
def word2bytes(word):
    "Convert word string to tuple of bytes"
    a = list(word.encode('utf-8'))
    return tuple(bytes([i]) for i in a)

In[16]:

In [ ]:
word2bytes('hello')

In[17]:

In [ ]:
word2bytes('€')

In[18]:

PAT = re.compile(r
'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+
)<br>
# In[19]:<br>
def count_word(text):<br>
    "Split text into word bytes using GPT2 pattern and count word bytes frequency."<br>
    word_cnt = defaultdict(int)<br>
    for m in PAT.finditer(text):<br>
        word = m.group(0)<br>
        word_bytes = word2bytes(word)<br>
        if len(word_bytes)>=2:<br>
            word_cnt[word_bytes]+=1<br>
    return word_cnt<br>
# In[20]:<br>
word_cnt = count_word(chunk)<br>
word_cnt<br>
# In[21]:<br>
def merge_dicts(dicts):<br>
    merged = defaultdict(int)<br>
    for d in dicts:<br>
        for k, v in d.items():<br>
            merged[k] += v<br>
    return merged<br>
# In[22]:<br>
merge_dicts([word_cnt,word_cnt])<br>
# In[23]:<br>
word_dicts =[count_word(chunk) for chunk in chunks]<br>
word_cnt_all = merge_dicts(word_dicts)<br>
# In[24]:<br>
word_cnt_all<br>
# In[25]:<br>
# parallel<br>
word_dicts = process_map(count_word, chunks,chunksize=1)<br>
# In[26]:<br>
word_cnt_all = merge_dicts(word_dicts)<br>
word_cnt_all<br>
# Note that some word only have single byte (len=1); no pair inside it<br>
# ## Step 3. Get pair count based on word count<br>
# In[27]:<br>
def count_pair(word_cnt):<br>
    pair_cnt = defaultdict(int)<br>
    for word_bytes,cnt in word_cnt.items():<br>
        for pair in zip(word_bytes[:-1],word_bytes[1:]):<br>
            pair_cnt[pair]+=cnt<br>
    return pair_cnt<br>
# In[28]:<br>
pair_cnt = count_pair(word_cnt_all)<br>
pair_cnt<br>
# ## Step 4. Get the max and merge<br>
# In[29]:<br>
def get_max_pair(pair_cnt): return max(pair_cnt.items(),key=lambda x: (x[1],x[0]))[0]<br>
# In[30]:<br>
max_pair = get_max_pair(pair_cnt)<br>
# In[31]:<br>
max_pair<br>
# We need to add the max_pair to both vocab and merges<br>
# In[32]:<br>
def get_basic_vocab(special_tokens):<br>
    vocab={token:bytes([token]) for token in range(256)}<br>
    for i,token in enumerate(special_tokens):<br>
        token_id = 256+i<br>
        vocab[token_id] = token.encode("utf-8")<br>
    return vocab<br>
# In[33]:<br>
special_tokens = ["<|endoftext|>", "<|pad|>", "<|unk|>"]<br>
# In[34]:<br>
vocab = get_basic_vocab(special_tokens)<br>
# In[35]:<br>
base_vocab_size = len(vocab)<br>
base_vocab_size<br>
# In[36]:<br>
vocab_size=270<br>
# In[37]:<br>
n_merges=vocab_size-len(vocab)<br>
n_merges<br>
# In[38]:<br>
def apply_merge(word_bytes,merge):<br>
    merged = merge[0]+merge[1]<br>
    i = 0<br>
    new_word_bytes = []<br>
    while i < len(word_bytes):<br>
        # Check for match<br>
        if i < len(word_bytes) - 1 and word_bytes[i] == merge[0] and word_bytes[i+1] == merge[1]:<br>
            new_word_bytes.append(merged)<br>
            i += 2<br>
        else:<br>
            new_word_bytes.append(word_bytes[i])<br>
            i += 1<br>
    return tuple(new_word_bytes)<br>
# In[39]:<br>
apply_merge(['a','b','c'],('b','c'))<br>
# In[40]:<br>
word_cnt<br>
# In[41]:<br>
def update_cnt(word_cnt,pair_cnt, merge_pair):<br>
    new_word_cnt = defaultdict(int)<br>
    new_pair_cnt = defaultdict(int, pair_cnt) # copy with defaultdict<br>
    for word_bytes,cnt in word_cnt.items():<br>
        #----------for word cnt ---------------<br>
        old_pairs = list(zip(word_bytes[:-1], word_bytes[1:]))<br>
        # Keep the original count if the merge not appear in the key<br>
        if merge_pair not in old_pairs:<br>
            new_word_cnt[word_bytes]+=cnt<br>
            continue<br>
        # Use updated key if merge appear<br>
        new_word = apply_merge(word_bytes,merge_pair)<br>
        new_word_cnt[new_word]+=cnt<br>
        #--------for pair cnt ----------------<br>
        # Decrease all old pair counts<br>
        for pair in old_pairs:<br>
            new_pair_cnt[pair]-=cnt<br>
            if new_pair_cnt[pair] ==0:<br>
                del new_pair_cnt[pair]<br>
        # Count new pairs in the new word<br>
        new_pairs = list(zip(new_word[:-1], new_word[1:]))<br>
        for p in new_pairs:<br>
            new_pair_cnt[p] += cnt<br>
    return new_word_cnt,new_pair_cnt<br>
# In[42]:<br>
word_cnt_new, pair_cnt_new = update_cnt(word_cnt_all,pair_cnt,max_pair)<br>
# In[43]:<br>
word_cnt_new<br>
# In[44]:<br>
pair_cnt_new<br>
# ## Pipeline<br>
# In[45]:<br>
test<br>
# In[46]:<br>
chunks = split_by_special(test,special_tokens)<br>
word_dicts = process_map(count_word, chunks,chunksize=1)<br>
word_cnt = merge_dicts(word_dicts)<br>
pair_cnt = count_pair(word_cnt)<br>
vocab = get_basic_vocab(special_tokens)<br>
base_vocab_size = len(vocab)<br>
vocab_size=265<br>
n_merges=vocab_size-base_vocab_size<br>
# In[47]:<br>
merges = []<br>
for i in range(n_merges):<br>
    max_pair = get_max_pair(pair_cnt)<br>
    vocab[base_vocab_size+i] = max_pair[0]+max_pair[1]<br>
    merges.append(max_pair)<br>
    word_cnt, pair_cnt = update_cnt(word_cnt,pair_cnt,max_pair)<br>
# In[48]:<br>
merges<br>
# In[49]:<br>
list(vocab.items())[base_vocab_size:]<br>
# Wrap them up in functions<br>
# In[50]:<br>
def read_text(input_path):<br>
    with open(input_path, "r", encoding="utf-8") as f:<br>
        text = f.read()<br>
    return text<br>
# In[51]:<br>
def train_bpe(input_path,vocab_size,special_tokens):<br>
  # PAT = r
'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+


In [ ]:
    text = read_text(input_path)
    chunks = split_by_special(text,special_tokens)
    word_dicts = process_map(count_word, chunks,chunksize=1)
    word_cnt = merge_dicts(word_dicts)
    pair_cnt = count_pair(word_cnt)
    vocab = get_basic_vocab(special_tokens)
    base_vocab_size = len(vocab)
    n_merges=vocab_size-base_vocab_size
    merges = []
    for i in range(n_merges):
        max_pair = get_max_pair(pair_cnt)
        vocab[base_vocab_size+i] = max_pair[0]+max_pair[1]
        merges.append(max_pair)
        word_cnt, pair_cnt = update_cnt(word_cnt,pair_cnt,max_pair)
    return vocab, merges

Put it in adapters.py and run `uv run pytest -k test_train_bpe`

In[52]:

In [ ]:
get_ipython().run_cell_magic('time', '', 'vocab,merges = train_bpe(input_path="corpus.en",\n                            vocab_size=1000,\n                            special_tokens=["<|endoftext|>"])\n')

In[53]:

In [ ]:
get_ipython().run_cell_magic('time', '', 'vocab,merges = train_bpe(input_path="corpus.en",\n                            vocab_size=1000,\n                            special_tokens=["<|endoftext|>"])\n')

In[54]:

In [ ]:
def build_occ_tables(word_cnt):
    """
    word_occ[word_id]   = (word_bytes, freq)
    pair_occ[pair]      = {word_id1, word_id2, ...}
    pair_freq[pair]     = 累计频数
    """
    word_occ = {}
    pair_occ = defaultdict(set)
    pair_freq = defaultdict(int)
    for wid, (wbytes, freq) in enumerate(word_cnt.items()):
        word_occ[wid] = (wbytes, freq)
        if len(wbytes) >= 2:
            for pair in zip(wbytes[:-1], wbytes[1:]):
                pair_occ[pair].add(wid)
                pair_freq[pair] += freq
    return word_occ, pair_occ, pair_freq

In[55]:

In [ ]:
word_occ, pair_occ, pair_freq = build_occ_tables(word_cnt)

In[56]:

In [ ]:
pair_freq

In[57]:

In [ ]:
pair_occ

In[58]:

In [ ]:
word_occ

In[59]:

In [ ]:
import regex as re
import heapq
from collections import defaultdict
from tqdm.contrib.concurrent import process_map

PAT_string = r
'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+
<br>
PAT = re.compile(PAT_string)<br>
# ---------- 基础工具 ----------<br>
def read_text(input_path: str) -> str:<br>
    with open(input_path, "r", encoding="utf-8") as f:<br>
        return f.read()<br>
def split_by_special(text, special_tokens, drop_special=True):<br>
    if not special_tokens:<br>
        return [text]<br>
    special_tokens = sorted(special_tokens, key=len, reverse=True)<br>
    pattern = "|".join(re.escape(tok) for tok in special_tokens)<br>
    pattern = f"({pattern})" if not drop_special else pattern<br>
    return [c for c in re.split(pattern, text) if c]<br>
def word2bytes(word: str):<br>
    return tuple(bytes([b]) for b in word.encode("utf-8"))<br>
def count_word(text: str):<br>
    cnt = defaultdict(int)<br>
    for m in PAT.finditer(text):<br>
        cnt[word2bytes(m.group(0))] += 1<br>
    return cnt<br>
def merge_dicts(dicts):<br>
    merged = defaultdict(int)<br>
    for d in dicts:<br>
        for k, v in d.items():<br>
            merged[k] += v<br>
    return merged<br>
def get_basic_vocab(special_tokens):<br>
    vocab = {tid: bytes([tid]) for tid in range(256)}<br>
    for i, tok in enumerate(special_tokens, start=256):<br>
        vocab[i] = tok.encode("utf-8")<br>
    return vocab<br>
def apply_merge(word_bytes, merge_pair):<br>
    merged_sym = merge_pair[0] + merge_pair[1]<br>
    out = []<br>
    i = 0<br>
    while i < len(word_bytes):<br>
        if i + 1 < len(word_bytes) and word_bytes[i:i+2] == list(merge_pair):<br>
            out.append(merged_sym)<br>
            i += 2<br>
        else:<br>
            out.append(word_bytes[i])<br>
            i += 1<br>
    return tuple(out)<br>
# ---------- 新增：occ 表 + 堆 ----------<br>
def build_occ_tables(word_cnt):<br>
  


In [ ]:
    word_occ[word_id]   = (word_bytes, freq)
    pair_occ[pair]      = {word_id1, word_id2, ...}
    pair_freq[pair]     = 累计频数
    """
    word_occ = {}
    pair_occ = defaultdict(set)
    pair_freq = defaultdict(int)
    for wid, (wbytes, freq) in enumerate(word_cnt.items()):
        word_occ[wid] = (wbytes, freq)
        if len(wbytes) >= 2:
            for pair in zip(wbytes[:-1], wbytes[1:]):
                pair_occ[pair].add(wid)
                pair_freq[pair] += freq
    return word_occ, pair_occ, pair_freq

In [ ]:
def build_heap(pair_freq):
    heap = [(-freq, pair) for pair, freq in pair_freq.items()]
    heapq.heapify(heap)
    return heap

In [ ]:
def get_max_pair_lazy(heap, pair_freq):
    """
    弹出堆顶，若与真实 freq 不符说明过期 -> 丢弃，继续。
    """
    while heap:
        neg_freq, pair = heap[0]
        if pair_freq[pair] != -neg_freq:
            heapq.heappop(heap)      # 过期
            continue
        return pair
    return None  # 所有 pair 都被合并完

In [ ]:
def update_after_merge(word_occ, pair_occ, pair_freq, heap, merge_pair):
    """
    只更新受 merge_pair 影响的 pretoken
    """
    affected = list(pair_occ[merge_pair])
    pair_occ[merge_pair].clear()

    # 真实频数用不到了，不再减；lazy 过期策略会自动淘汰旧值
    for wid in affected:
        wbytes, freq = word_occ[wid]

        # 1) 先把旧 word 的所有 pair 在 occ 表里注销
        if len(wbytes) >= 2:
            for p in zip(wbytes[:-1], wbytes[1:]):
                if wid in pair_occ[p]:
                    pair_occ[p].discard(wid)

        # 2) 生成新 word，并重新登记
        new_wbytes = apply_merge(wbytes, merge_pair)
        word_occ[wid] = (new_wbytes, freq)
        if len(new_wbytes) >= 2:
            for p in zip(new_wbytes[:-1], new_wbytes[1:]):
                pair_occ[p].add(wid)
                pair_freq[p] += freq          # 只加不减
                heapq.heappush(heap, (-pair_freq[p], p))

---------- 训练主函数 ----------

In [ ]:
def train_bpe(input_path, vocab_size, special_tokens=()):
    text = read_text(input_path)
    chunks = split_by_special(text, special_tokens)

    # 计数可以并行
    word_dicts = (
        process_map(count_word, chunks, chunksize=1)
        if len(chunks) >= 4
        else list(map(count_word, chunks))
    )
    word_cnt = merge_dicts(word_dicts)

    # === 初始化 occ & heap ===
    word_occ, pair_occ, pair_freq = build_occ_tables(word_cnt)
    heap = build_heap(pair_freq)

    # === 初始化 vocab ===
    vocab = get_basic_vocab(special_tokens)
    base = len(vocab)
    merges_needed = vocab_size - base
    merges = []
    for i in range(merges_needed):
        best_pair = get_max_pair_lazy(heap, pair_freq)
        if best_pair is None:
            break  # 提前终止：再也找不到可合并的 pair
        merges.append(best_pair)
        vocab[base + i] = best_pair[0] + best_pair[1]
        update_after_merge(word_occ, pair_occ, pair_freq, heap, best_pair)
    return vocab, merges

## Encode

In[60]:

In [ ]:
special_tokens

In[61]:

PAT = r
'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+
<br>
# In[62]:<br>
chunks = split_by_special(test,special_tokens,drop_special=False)<br>
chunks<br>
# In[63]:<br>
chunk=chunks[0]<br>
# In[64]:<br>
def split_to_words(text):<br>
    "Split text into words."<br>
    return re.findall(PAT,text)<br>
# In[65]:<br>
word_list = split_to_words(chunk)<br>
word_list<br>
# In[66]:<br>
vocab_to_id = {v:k for k,v in vocab.items()}<br>
# In[67]:<br>
def apply_merges(word_bytes, merges, vocab_to_id):<br>
    "Apply merges based on minimum vocab token id."<br>
    while True:<br>
        pairs = list(zip(word_bytes[:-1], word_bytes[1:]))<br>
        # Collect valid merge candidates with their vocab ID<br>
        candidates = {}<br>
        for pair in pairs:<br>
            if pair in merges:<br>
                merged = pair[0] + pair[1]<br>
                token_id = vocab_to_id.get(merged)<br>
                if token_id is not None:<br>
                    candidates[pair] = token_id<br>
        if not candidates:<br>
            break  # no more mergeable pairs<br>
        # Choose the pair with the **smallest token ID**<br>
        best_pair = min(candidates.items(), key=lambda x: x[1])[0]<br>
        word_bytes = apply_merge(word_bytes, best_pair)<br>
    return word_bytes<br>
# In[68]:<br>
word_bytes=word2bytes(' world')<br>
word_bytes<br>
# In[ ]:<br>
merged_word_bytes = apply_merges(word_bytes,merges,vocab_to_id)<br>
merged_word_bytes<br>
# In[ ]:<br>
def encode_merged(text,merges,vocab_to_id):<br>
    word_list = split_to_words(text)<br>
    tokens=[]<br>
    for word in word_list:<br>
        word_bytes=word2bytes(word)<br>
        merged_word_bytes = apply_merges(word_bytes,merges,vocab_to_id)<br>
        tokens+=[vocab_to_id[i] for i in merged_word_bytes]<br>
    return tokens<br>
# In[ ]:<br>
encode_merged(chunk,merges,vocab_to_id)<br>
# In[ ]:<br>
chunks = split_by_special(test,special_tokens,drop_special=False)<br>
tokens =[]<br>
for chunk in chunks:<br>
    if chunk in special_tokens:<br>
        tokens+=[vocab_to_id[chunk.encode('utf-8')]]<br>
    else:<br>
        tokens+=encode_merged(chunk,merges,vocab_to_id)<br>
# In[ ]:<br>
print(tokens)<br>
# ## Decode<br>
# In[ ]:<br>
def decode(tokens,vocab): return b''.join([vocab[t] for t in tokens]).decode('utf-8',errors='replace')<br>
# In[ ]:<br>
decode(tokens,vocab)<br>
# In[ ]:<br>
# if not indicate errors=replace, will throw an error if decode([128],vocab) as 128 unicode is 10... not start bytes<br>
# In[ ]:<br>
decode([128],vocab) # this time will throw a question mark<br>
# ## Class tokenizer<br>
# In[ ]:<br>
from typing import Iterator, Iterable<br>
import json<br>
# In[ ]:<br>
class Tokenizer:<br>
    def __init__(self, vocab, merges, special_tokens=None):<br>
        self.vocab = vocab<br>
        self.merges = merges<br>
        self.special_tokens = special_tokens if special_tokens else []<br>
        self.special_tokens_bytes = [i.encode('utf-8') for i in self.special_tokens]<br>
        self.vocab_to_id={v:k for k,v in vocab.items()}<br>
        # Ensure special tokens are in the vocabulary<br>
        for token_bytes in self.special_tokens_bytes:<br>
            if token_bytes not in self.vocab_to_id:<br>
                # Add to vocab if not already present<br>
                new_id = len(self.vocab)<br>
                self.vocab[new_id] = token_bytes<br>
                self.vocab_to_id[token_bytes] = new_id<br>
    @classmethod<br>
    def from_files(cls, vocab_filepath, merges_filepath, special_tokens=None):<br>
        # Load vocab (assumed to be a JSON file: {token_id: byte_string})<br>
        with open(vocab_filepath, 'r', encoding='utf-8') as vf:<br>
            vocab_data = json.load(vf)<br>
            # Optional: convert keys to int if stored as strings<br>
            vocab = {int(k): bytes(v, 'latin1') if isinstance(v, str) else bytes(v)<br>
                     for k, v in vocab_data.items()}<br>
        # Load merges (assumed to be a list of pairs like: "a b")<br>
        with open(merges_filepath, 'r', encoding='utf-8') as mf:<br>
            lines = mf.readlines()<br>
            # Optional: skip headers like "#version: 0.2"<br>
            merge_pairs = [tuple(line.strip().split()) for line in lines if not line.startswith('#') and line.strip()]<br>
            # Convert to byte-pairs<br>
            merges = [(a.encode('utf-8'), b.encode('utf-8')) for a, b in merge_pairs]<br>
        return cls(vocab=vocab, merges=merges, special_tokens=special_tokens)<br>
    def encode(self, text: str) -> list[int]:<br>
        chunks = split_by_special(text, self.special_tokens, drop_special=False)<br>
        tokens = []<br>
        for chunk in chunks:<br>
            if self.special_tokens and chunk in self.special_tokens:<br>
                tokens.append(self.vocab_to_id[chunk.encode('utf-8')])<br>
            else:<br>
                tokens.extend(encode_merged(chunk, self.merges, self.vocab_to_id))<br>
        return tokens<br>
    def encode_iterable(self, iterable: Iterable[str]) -> Iterator[int]:<br>
      


In [ ]:
        Given an iterable of strings (e.g., a Python file handle), return a generator that lazily yields token IDs.
        This is required for memory-efficient tokenization of large files that we cannot directly load into memory.
        """
        for chunk in iterable:
            yield from self.encode(chunk)
    def decode(self, ids: list[int]) -> str:
        "Decode a sequence of token IDs into text."
        return b''.join([self.vocab[t] for t in ids]).decode('utf-8',errors='replace')

In[ ]:

In [ ]:
import regex as re
from collections import defaultdict
import heapq
from tqdm.contrib.concurrent import process_map

In[ ]:

In [ ]:
a = defaultdict(set)

In[ ]:

In [ ]:
a['a'].add('b')

In[ ]:

In [ ]:
a

In[ ]: